# Online Retail II – Data Cleaning
Loading, exploring and cleaning the raw transaction data before loading it into MySQL.

In [19]:
import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)

In [4]:
sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

In [5]:
df.columns = ["invoice", "stock_code", "description", "quantity",
              "invoice_date", "price", "customer_id", "country"]

print(df.shape)
df.head()

(1067371, 8)


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  object        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[us]
 5   price         1067371 non-null  float64       
 6   customer_id   824364 non-null   float64       
 7   country       1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [7]:
df.isna().sum()

invoice              0
stock_code           0
description       4382
quantity             0
invoice_date         0
price                0
customer_id     243007
country              0
dtype: int64

First I explored the data and found that about 22.7% of rows had no customer ID

4,382 rows have no product description

In [8]:
start_rows = len(df)
print("Rows before cleaning:", start_rows)

Rows before cleaning: 1067371


In [9]:
df["invoice"] = df["invoice"].astype(str)
df["stock_code"] = df["stock_code"].astype(str)
df["customer_id"] = df["customer_id"].astype("Int64")

df.dtypes

invoice                    str
stock_code                 str
description             object
quantity                 int64
invoice_date    datetime64[us]
price                  float64
customer_id              Int64
country                    str
dtype: object

Converted invoice and stock code to text and customer ID to whole numbers

In [10]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Rows now:", len(df))

Duplicate rows: 34335
Rows now: 1033036


Removed 34335 exact duplicate rows to avoid double-counting sales.

In [11]:
df["is_cancelled"] = df["invoice"].str.startswith("C")
df["is_cancelled"].value_counts()

is_cancelled
False    1013932
True       19104
Name: count, dtype: int64

Flagged 19104 cancelled rows instead of deleting them, so cancellation rates can be analysed later

In [12]:
print("Rows with price <= 0:", (df["price"] <= 0).sum())
df[df["price"] <= 0].head(10)

Rows with price <= 0: 6019


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_cancelled
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom,False
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,False
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,False
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,<NA>,United Kingdom,False
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,<NA>,United Kingdom,False
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,<NA>,United Kingdom,False
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,<NA>,United Kingdom,False
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom,False


In [13]:
df = df[df["price"] > 0]
print("Rows now:", len(df))

Rows now: 1027017


Removed 6019 rows with zero or negative price 

In [14]:
odd_codes = df[~df["stock_code"].str.match(r"^\d{5}")]
odd_codes.groupby("stock_code")["description"].first().head(40)

stock_code
ADJUST          Adjustment by john on 26/01/2010 16
ADJUST2         Adjustment by Peter on Jun 25 2010 
AMAZONFEE                                AMAZON FEE
B                                   Adjust bad debt
BANK CHARGES                           Bank Charges
C2                                         CARRIAGE
CRUK                                CRUK Commission
D                                          Discount
DCGS0003                        BOXED GLASS ASHTRAY
DCGS0004                 HAYNES CAMPER SHOULDER BAG
DCGS0037                         KEY-RING CORKSCREW
DCGS0041           HAYNES MINI-COOPER PLAYING CARDS
DCGS0044                    HANDZ-OFF CAR FRESHENER
DCGS0058                           MISO PRETTY  GUM
DCGS0062                    ROAD-RAGE CAR FRESHENER
DCGS0066N                   NAVY CUDDLES DOG HOODIE
DCGS0068                          DOGS NIGHT COLLAR
DCGS0069                      OOH LA LA DOGS COLLAR
DCGS0070                      CAMOUFLAGE DOG COLLAR
D

In [15]:
non_products = ["POST", "DOT", "C2",
                "AMAZONFEE", "BANK CHARGES", "CRUK",
                "ADJUST", "ADJUST2", "B", "D", "M", "m",
                "S", "TEST001", "TEST002"]

is_non_product = (
    df["stock_code"].isin(non_products)
    | df["stock_code"].str.startswith("gift_")
)

print("Rows to remove:", is_non_product.sum())
df = df[~is_non_product]
print("Rows now:", len(df))

Rows to remove: 5745
Rows now: 1021272


Removed non-product codes: postage, fees, adjustments, discounts, samples, test entries and gift vouchers. Kept codes with a non-standard format that are real products (e.g. DCGS items, PADS, SP1002)

In [16]:
df["revenue"] = df["quantity"] * df["price"]
df[["quantity", "price", "revenue"]].head()

,quantity,price,revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [17]:
print("Rows before:", start_rows)
print("Rows after:", len(df))
print("Removed:", start_rows - len(df), f"({(start_rows - len(df)) / start_rows:.1%})")
print()
print(df.isna().sum())

Rows before: 1067371
Rows after: 1021272
Removed: 46099 (4.3%)

invoice              0
stock_code           0
description          0
quantity             0
invoice_date         0
price                0
customer_id     227089
country              0
is_cancelled         0
revenue              0
dtype: int64


In [20]:
df.describe()

,quantity,invoice_date,price,customer_id,revenue
count,"1,021,272.00",1021272,"1,021,272.00","794,183.00","1,021,272.00"
mean,10.50,2011-01-04 01:29:56.340211,3.36,"15,319.71",18.53
min,"-80,995.00",2009-12-01 07:45:00,0.00,"12,346.00","-168,469.60"
25%,1.00,2010-07-05 14:41:00,1.25,"13,971.00",3.75
50%,3.00,2010-12-09 14:09:00,2.10,"15,244.00",9.95
75%,10.00,2011-07-28 09:19:00,4.13,"16,791.00",17.70
max,"80,995.00",2011-12-09 12:50:00,"1,157.15","18,287.00","168,469.60"
std,169.08,NaN,4.87,"1,693.61",270.61


In [21]:
cols = ["invoice", "stock_code", "description", "quantity", "price", "invoice_date", "customer_id", "is_cancelled"]
df[df["quantity"].abs() > 10000][cols].sort_values("quantity")

,invoice,stock_code,description,quantity,price,invoice_date,customer_id,is_cancelled
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00,16446,True
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,2011-01-18 10:17:00,12346,True
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,2010-03-17 13:09:00,13902,False
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,2010-03-17 13:09:00,13902,False
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,2010-03-17 13:09:00,13902,False
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,2010-03-17 13:09:00,13902,False
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,2010-02-15 11:57:00,13902,False
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,2011-01-18 10:01:00,12346,False
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00,16446,False


In [22]:
error_pairs = (
    df["invoice"].isin(["581483", "C581484", "541431", "C541433"])
    & df["stock_code"].isin(["23843", "23166"])
)

print("Rows to remove:", error_pairs.sum())
df = df[~error_pairs]
print("Rows now:", len(df))

Rows to remove: 4
Rows now: 1021268


In [23]:
df["quantity"].describe()

count   1,021,268.00
mean           10.50
std            70.39
min        -9,360.00
25%             1.00
50%             3.00
75%            10.00
max        19,152.00
Name: quantity, dtype: float64

removed two erroneous order–cancellation pairs (invoices 581483/C581484 and 541431/C541433): each was an extreme quantity (80,995 and 74,215 units) cancelled within minutes with the identical quantity. Kept other large non-cancelled wholesale orders (e.g. customer 13902), as they appear to be genuine bulk purchases.